In [ ]:
import pandas as pd
import geopandas as gpd

In [ ]:
## data prep for open day
#read in the places, shapefile and racialsed reviews
places_file = '/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/places_filtered.parquet'
shapefile_file = '/home/ethel/Desktop/swimming_pool_project/btw25_geometrie_wahlkreise_shp/btw25_geometrie_wahlkreise_shp.shp'
racialised_reviews_file = '/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/racialised_reclassed_reviews.parquet'
all_reviews_file = '/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/social_classified_text_reviews.parquet'
demographics_file = '/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/constituency_strukturdaten_28112025.csv'


In [ ]:
places_df = pd.read_parquet(places_file)
shapefile_gdf = gpd.read_file(shapefile_file)
racialised_reviews_df = pd.read_parquet(racialised_reviews_file)
all_reviews_df = pd.read_parquet(all_reviews_file)
demographics_df = pd.read_csv(demographics_file)

In [ ]:
places_df.info()

In [ ]:
places_df[['unique_identifier', 'place_name', 'latitude','longitude', 'address']].to_csv('/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/places_filtered_coords.csv', index=False)

In [ ]:
racialised_reviews_df[racialised_reviews_df['review_id'] == 'ChZDSUhNMG9nS0VJQ0FnSUNiLTRyVElnEAE']

In [ ]:
racialised_reviews_df.info()

In [ ]:
places_df[places_df['unique_identifier'] == '0x47a4288568a92fc7:0xb4365f9e11324ad2']

In [ ]:
##create the new total_text_reviews column from  the all text reviews column in the all_reviews_df
#all_reviews_df['total_text_reviews'] = 
all_reviews_df['total_text_reviews'] = (
    all_reviews_df
    .groupby('unique_identifier')['review_id']
    .transform('count')
)


In [ ]:
all_reviews_df.shape

In [ ]:
#select columns from places and racialised reviews to merge on
places_cols = ['unique_identifier', 'place_id', 'latitude', 'longitude']

all_reviews_cols = ['review_id', 'unique_identifier', 'reviewer_id', 'social', 'total_text_reviews']
                          
racialised_reviews_cols = ['review_id', 'unique_identifier', 'reviewer_id', 'racialised']

demographic_cols = ['state_name', 'constituency_number', 'constituency_name',]


In [ ]:
#rename columns for clarity
shapefile_gdf = shapefile_gdf.rename(columns={'WKR_NR': 'constituency_number', 'WKR_NAME': 'constituency_name', 
                                              'LAND_NR': 'state_number', 'LAND_NAME': 'state_name'})

In [ ]:
#merge places and racialised reviews on unique identifier
merged_df = pd.merge(all_reviews_df[all_reviews_cols], racialised_reviews_df[racialised_reviews_cols], on=['review_id', 'unique_identifier', 'reviewer_id'], how='outer')
merged_df = pd.merge(merged_df,
                      places_df[places_cols], on='unique_identifier', how='outer')
merged_df.info()

In [ ]:
# # Step 1: Reproject
# shapefile_gdf = shapefile_gdf.to_crs(epsg=3857)
# merged_gdf = gpd.GeoDataFrame(
#     merged_df,
#     geometry=gpd.points_from_xy(merged_df.longitude, merged_df.latitude),
#     crs='EPSG:4326'
# ).to_crs(epsg=3857)

# print("=== DIAGNOSTICS ===")
# print(f"merged_gdf points: {len(merged_gdf)}")
# print(f"shapefile polygons: {len(shapefile_gdf)}")
# print(f"shapefile CRS: {shapefile_gdf.crs}")
# print(f"merged_gdf CRS: {merged_gdf.crs}")
# print(f"shapefile geometry types: {shapefile_gdf.geometry.geom_type.value_counts().to_dict()}")
# print(f"any null geometries in points: {merged_gdf.geometry.isna().sum()}")
# print(f"any null geometries in shapefile: {shapefile_gdf.geometry.isna().sum()}")
# print(f"any invalid geometries in shapefile: {(~shapefile_gdf.geometry.is_valid).sum()}")

# # Check bounding boxes overlap
# print(f"\nPoints bounds:    {merged_gdf.total_bounds}")
# print(f"Shapefile bounds: {shapefile_gdf.total_bounds}")

# # Step 2: Dissolved filter
# dissolved = shapefile_gdf.geometry.union_all()
# shapefile_for_filter = gpd.GeoDataFrame(geometry=[dissolved], crs=shapefile_gdf.crs)
# shapefile_for_filter['geometry'] = shapefile_for_filter['geometry'].buffer(50000)

# print(f"\nDissolved geometry type: {shapefile_for_filter.geometry.geom_type[0]}")
# print(f"Dissolved geometry valid: {shapefile_for_filter.geometry.is_valid[0]}")

# # Step 3: Filter
# points_in_boundary = gpd.sjoin(
#     merged_gdf, shapefile_for_filter, how='inner', predicate='within'
# ).drop(columns=['index_right'])
# print(f"\nAfter dissolved filter: {len(points_in_boundary)}")

# # Step 4: nearest join
# spatial_join_gdf = gpd.sjoin_nearest(points_in_boundary, shapefile_gdf, how='left')
# print(f"After sjoin_nearest: {len(spatial_join_gdf)}")

# spatial_join_gdf = spatial_join_gdf.drop_duplicates(subset='review_id')
# spatial_join_gdf = spatial_join_gdf.drop(columns=['index_right'])
# print(f"After dedup: {len(spatial_join_gdf)}")

In [ ]:
#import geopandas as gpd

# Step 1: Reproject
shapefile_gdf = shapefile_gdf.to_crs(epsg=3857)
merged_gdf = gpd.GeoDataFrame(
    merged_df,
    geometry=gpd.points_from_xy(merged_df.longitude, merged_df.latitude),
    crs='EPSG:4326'
).to_crs(epsg=3857)

# Step 2: Dissolve into one MultiPolygon — NO buffer
# This preserves all islands as separate polygons within the MultiPolygon
dissolved = shapefile_gdf.geometry.union_all()
germany_exact = gpd.GeoDataFrame(geometry=[dissolved], crs=shapefile_gdf.crs)

# Step 3: Strict containment — point must be inside actual Germany geometry
points_in_germany = gpd.sjoin(
    merged_gdf, germany_exact, how='inner', predicate='within'
).drop(columns=['index_right'])

print(f"Points strictly inside Germany (incl. islands): {len(points_in_germany)}")

# Step 4: Nearest join to assign attributes from the shapefile
spatial_join_gdf = gpd.sjoin_nearest(points_in_germany, shapefile_gdf, how='left')
spatial_join_gdf = spatial_join_gdf.drop_duplicates(subset='review_id')
spatial_join_gdf = spatial_join_gdf.drop(columns=['index_right'])

print(f"Final deduplicated count: {len(spatial_join_gdf)}")

In [ ]:
spatial_join_gdf.info()

In [ ]:
spatial_join_gdf.unique_identifier.nunique()

In [ ]:
all_reviews_df[all_reviews_df['social'] == 'yes'].info()

In [ ]:
#add the east/west variable
def classify_east_west(row):
    if row['state_name'] in ['Brandenburg', 'Mecklenburg-Vorpommern', 'Sachsen', 'Sachsen-Anhalt', 'Thüringen', 'Berlin']:
        return 'east'
    else: 
        return 'west'
spatial_join_gdf['east_west'] =  spatial_join_gdf.apply(classify_east_west, axis=1)

## Aggregation

In [ ]:

# --- Constituency-level aggregation ---
constituency_agg = (
    spatial_join_gdf.groupby(
        ["constituency_number", "constituency_name", "state_number", "state_name", "east_west"],
        as_index=False
    )
    .agg(
        pool_count=("place_id", "nunique"),
        total_reviews=("review_id", "nunique"),
        social_reviews=("social", lambda x: (x == "yes").sum()),
        racialised_reviews=("racialised", lambda x: (x == "yes").sum())
    )
)

# --- Merge aggregated data back with geometry ---
constituency_geo = (
    spatial_join_gdf[["constituency_number", "geometry"]]
    .dissolve(by="constituency_number")
    .reset_index()
    .merge(constituency_agg, on="constituency_number")
)

# --- Save both ---
constituency_agg.to_csv("/home/ethel/Desktop/swimming_pool_project/spp_15052026/openday_2026/constituency_aggregated.csv", index=False)
constituency_geo.to_file("/home/ethel/Desktop/swimming_pool_project/spp_15052026/openday_2026/constituency_aggregated.gpkg", driver="GPKG")

print(f"Constituencies: {len(constituency_agg)}")
print(f"Total reviews: {constituency_agg['total_reviews'].sum():,}")
print(f"Social reviews: {constituency_agg['social_reviews'].sum():,}")
print(f"Racialised reviews: {constituency_agg['racialised_reviews'].sum():,}")
print(f"Total pools: {constituency_agg['pool_count'].sum():,}")
print(constituency_agg.head())

In [ ]:
# load the new classification
approach_file = '/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/reclassed_racialised_approach.parquet'
themes_file = '/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/reclassed_racialised_theme.parquet'
connotation_file = '/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/reclassed_racialised_connotation.parquet'
context_file = '/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/reclassed_racialised_context.parquet'

#read all the files merge and explode them
theme_df = pd.read_parquet(themes_file)
context_df = pd.read_parquet(context_file)
connotation_df = pd.read_parquet(connotation_file)
approach_df = pd.read_parquet(approach_file)


racialised_dimensions = theme_df[['review_id', 'theme']].merge(context_df[['review_id', 'context']], on='review_id', how='outer')\
    .merge(connotation_df[['review_id', 'connotation']], on='review_id', how='outer')\
    .merge(approach_df[['review_id', 'approach']], on='review_id', how='outer')
    # drop null rows
racialised_dimensions = racialised_dimensions.dropna(subset=['theme', 'context', 'connotation', 'approach'], how='any').reset_index(drop=True) 

In [ ]:
# This is not valid JSON (no commas) so we parse it manually
def parse_list(val):
    if pd.isna(val):
        return []
    # strip brackets, split on whitespace, clean quotes
    val = str(val).strip().strip("[]")
    items = [v.strip().strip("'\"") for v in val.split("'") if v.strip().strip("'\"") != ',']
    #make lowercase
    items = [i.lower() for i in items]
    return [i for i in items if i]  # drop empty strings


In [ ]:
#parse and explode
for col in ['theme', 'context', 'connotation', 'approach']:
    racialised_dimensions[col] = racialised_dimensions[col].apply(parse_list)
    racialised_dimensions = racialised_dimensions.explode(col, ignore_index=True)

In [ ]:
racialised_dimensions_augmented = racialised_dimensions[['review_id', 'theme', 'context', 'connotation', 'approach']].merge(
 spatial_join_gdf[['review_id', 'state_name', 'constituency_name', 'east_west']], on='review_id', how='left')

In [ ]:
racialised_dimensions_augmented = racialised_dimensions_augmented[racialised_dimensions_augmented['review_id'].isin(racialised_reviews_df['review_id'])]
racialised_dimensions_augmented.info()

In [ ]:
racialised_dimensions_augmented.to_csv(("/home/ethel/Desktop/swimming_pool_project/spp_15052026/openday_2026/racialised_dimensions_spaitial_exploded.csv"), index=False)

In [ ]:
# Check how many unique combinations per review
racialised_dimensions_augmented.groupby("review_id").size().value_counts().sort_index()

In [ ]:
# Check per dimension how many reviews have multiple values
for col in ["theme", "connotation", "approach", "context"]:
    print(f"\n{col}:")
    print(racialised_dimensions_augmented.groupby("review_id")[col].nunique().value_counts().sort_index())

In [ ]:
import pandas as pd

# --- Theme (multi-valued, count all unique theme instances per review) ---
theme_agg = (
    racialised_dimensions_augmented.drop_duplicates(subset=["review_id", "theme"])
    .groupby("theme", as_index=False)
    .agg(count=("review_id", "count"))
    .sort_values("count", ascending=False)
)

# --- Connotation (mostly single, occasionally 2) ---
connotation_agg = (
    racialised_dimensions_augmented.drop_duplicates(subset=["review_id", "connotation"])
    .groupby("connotation", as_index=False)
    .agg(count=("review_id", "count"))
)

# --- Approach (always single) ---
approach_agg = (
    racialised_dimensions_augmented.drop_duplicates(subset=["review_id", "approach"])
    .groupby("approach", as_index=False)
    .agg(count=("review_id", "count"))
)

# --- Context (1 or 2 values) ---
context_agg = (
    racialised_dimensions_augmented.drop_duplicates(subset=["review_id", "context"])
    .groupby("context", as_index=False)
    .agg(count=("review_id", "count"))
)

# --- East/West breakdowns ---
theme_east_west = (
    racialised_dimensions_augmented.drop_duplicates(subset=["review_id", "theme"])
    .groupby(["east_west", "theme"], as_index=False)
    .agg(count=("review_id", "count"))
)

connotation_east_west = (
    racialised_dimensions_augmented.drop_duplicates(subset=["review_id", "connotation"])
    .groupby(["east_west", "connotation"], as_index=False)
    .agg(count=("review_id", "count"))
)

# --- State-level theme breakdown ---
theme_state = (
    racialised_dimensions_augmented.drop_duplicates(subset=["review_id", "theme", "state_name"])
    .groupby(["state_name", "theme"], as_index=False)
    .agg(count=("review_id", "count"))
)

# --- Save all ---
theme_agg.to_csv("/home/ethel/Desktop/swimming_pool_project/spp_15052026/openday_2026/agg_theme.csv", index=False)
connotation_agg.to_csv("/home/ethel/Desktop/swimming_pool_project/spp_15052026/openday_2026/agg_connotation.csv", index=False)
approach_agg.to_csv("/home/ethel/Desktop/swimming_pool_project/spp_15052026/openday_2026/agg_approach.csv", index=False)
context_agg.to_csv("/home/ethel/Desktop/swimming_pool_project/spp_15052026/openday_2026/agg_context.csv", index=False)
theme_east_west.to_csv("/home/ethel/Desktop/swimming_pool_project/spp_15052026/openday_2026/agg_theme_east_west.csv", index=False)
connotation_east_west.to_csv("/home/ethel/Desktop/swimming_pool_project/spp_15052026/openday_2026/agg_connotation_east_west.csv", index=False)
theme_state.to_csv("/home/ethel/Desktop/swimming_pool_project/spp_15052026/openday_2026/agg_theme_state.csv", index=False)

# --- Sanity checks --- all should be <= 7,768 per label, totals can exceed 7,768 ---
print(f"Theme total instances: {theme_agg['count'].sum():,} (can exceed 7,768)")
print(f"Connotation total: {connotation_agg['count'].sum():,} (can slightly exceed 7,768)")
print(f"Approach total: {approach_agg['count'].sum():,} (should equal 7,768)")
print(f"Context total: {context_agg['count'].sum():,} (can slightly exceed 7,768)")
print(f"\nThemes:\n{theme_agg}")
print(f"\nConnotation:\n{connotation_agg}")
print(f"\nApproach:\n{approach_agg}")
print(f"\nContext:\n{context_agg}")

In [ ]:
import pandas as pd

# --- assume your exploded df, and also your original racialised reviews ---
# We'll sample at the REVIEW level first, then pull their text from the original

# Step 1: get one row per review (deduplicated) with all dimension labels
# since theme is multi-valued, we'll take the first theme per review for sampling purposes
review_level = (
    racialised_dimensions_augmented.drop_duplicates(subset=["review_id"])
    [["review_id", "connotation", "approach", "context", "state_name", "east_west"]]
)

# Step 2: stratified sample by connotation (most important for balance)
# proportional but with a floor to ensure positive/neutral are represented
samples = []

targets = {
    "negative": 12,   # majority but not overwhelming
    "neutral":   7,   # oversampled relative to its share
    "positive":  6,   # oversampled relative to its share
}

for connotation, n in targets.items():
    pool = review_level[review_level["connotation"] == connotation]
    n = min(n, len(pool))  # in case fewer exist
    samples.append(pool.sample(n, random_state=42))

sampled = pd.concat(samples).reset_index(drop=True)
sampled = pd.merge(sampled, racialised_reviews_df[['review_id','org_review', 'trans_review', 'org_review_lang']], on='review_id', how='left')

# Step 3: show the sample with all dimensions for review
print(f"Total sampled: {len(sampled)}")
print(sampled[["review_id", "org_review", "trans_review", "org_review_lang", "connotation", "approach", "context", "state_name", "east_west"]])

# Step 4: save so you can manually add the review text
sampled.to_csv("/home/ethel/Desktop/swimming_pool_project/spp_15052026/openday_2026/review_samples_to_fill.csv", index=False)

In [ ]:
racialised_reviews_df.info()

In [ ]:
racialised_reviews_df[['review_id','org_review', 'trans_review', 'org_review_lang', 'review_time', 'retrieved_time']].to_csv("/home/ethel/Desktop/swimming_pool_project/spp_15052026/openday_2026/racialised_reviews_cols.csv", index=False)

In [ ]:
print(racialised_reviews_df.groupby('place_id')['unique_identifier'].nunique().describe())
print(f"Unique place_ids: {racialised_reviews_df['place_id'].nunique()}")
print(f"Unique unique_identifiers: {racialised_reviews_df['unique_identifier'].nunique()}")

In [ ]:
racialised_reviews_df.info()

In [ ]:
df = racialised_reviews_df.copy()

# Are the missing retrieved_time reviews clustered by place_id or time?
missing_rt = df[df['retrieved_time'].isna()]
has_rt = df[df['retrieved_time'].notna()]

print("Missing retrieved_time:", len(missing_rt))
print("Has retrieved_time:", len(has_rt))

# Do missing ones have review_time?
print("\nOf missing retrieved_time:")
print("  Has review_time:", missing_rt['review_time'].notna().sum())
print("  Missing review_time too:", missing_rt['review_time'].isna().sum())

# Distribution of review_time strings for missing vs present
print("\nreview_time samples (missing retrieved_time):")
print(missing_rt['review_time'].dropna().value_counts().head(15))

print("\nretrieved_time range (where present):")
print(has_rt['retrieved_time'].min(), "→", has_rt['retrieved_time'].max())

# Are missing ones from specific states?
# (need to join with main gdf by review_id first)

In [ ]:
import pandas as pd

BASE = "/home/ethel/Desktop/swimming_pool_project/spp_15052026/openday_2026/"

dim  = pd.read_csv(BASE + "racialised_dimensions_spaitial_exploded.csv")
time = pd.read_csv(BASE + "reviews_with_years.csv")[["review_id", "period"]]

# One row per review
deduped = (
    dim.drop_duplicates(subset="review_id", keep="first")
    [["review_id", "state_name", "connotation", "approach", "context"]]
    .merge(time, on="review_id", how="left")
)

# Normalise
for col in ["connotation", "approach", "context"]:
    deduped[col] = deduped[col].str.strip().str.lower()

# Drop rows where period is missing
deduped = deduped.dropna(subset=["period"])
print("deduped:", deduped.shape)

# Binary indicators
deduped["is_explicit"]    = deduped["approach"].isin(["explicit", "both"]).astype(int)
deduped["is_generalised"] = (deduped["context"] == "generalised").astype(int)
deduped["is_negative"]    = (deduped["connotation"] == "negative").astype(int)

# Aggregate: state × period
agg = (
    deduped
    .groupby(["state_name", "period"])
    .agg(
        count           = ("review_id",       "count"),
        explicit_pct    = ("is_explicit",     "mean"),
        generalised_pct = ("is_generalised",  "mean"),
        negative_pct    = ("is_negative",     "mean"),
    )
    .reset_index()
)

# Round percentages
for col in ["explicit_pct", "generalised_pct", "negative_pct"]:
    agg[col] = (agg[col] * 100).round(1)

# Add east/west
ew  = dim[["state_name", "east_west"]].drop_duplicates("state_name")
agg = agg.merge(ew, on="state_name", how="left")

print("agg:", agg.shape)
print(agg.head(12))

agg.to_csv(BASE + "scatter_state_period.csv", index=False)
print("Done — upload scatter_state_period.csv")

## getting some extreem reviews

In [ ]:
import pandas as pd

BASE = "/home/ethel/Desktop/swimming_pool_project/spp_15052026/openday_2026/"

# Load the unexploded file (one row per review)
dim  = pd.read_csv(BASE + "racialised_dimensions_spaitial_exploded.csv")
time = pd.read_csv(BASE + "reviews_with_years.csv")[["review_id", "period"]]

# One row per review — drop duplicates from exploded file
deduped = (
    dim.drop_duplicates(subset="review_id", keep="first")
    [["review_id", "state_name", "connotation", "approach", "context", "east_west"]]
    .merge(time, on="review_id", how="left")
)

# Normalise
for col in ["connotation", "approach", "context"]:
    deduped[col] = deduped[col].str.strip().str.lower()

# Filter: generalised context only
generalised = deduped[deduped["context"] == "generalised"].copy()

# Join the actual review text
# You'll need the file that has review_id + review text
# Adjust the filename/column name to match yours
reviews = pd.read_csv(BASE + "racialised_reviews_cols.csv")[["review_id", "org_review", "trans_review"]]
generalised = generalised.merge(reviews, on="review_id", how="left")

# Filter: short reviews (under 100 chars in original text)
generalised["text_len"] = generalised["org_review"].fillna("").str.len()
short_gen = generalised[generalised["text_len"] <= 100].copy()

# Sort by length ascending — shortest first
short_gen = short_gen.sort_values("text_len")

# Drop rows with no text
short_gen = short_gen[short_gen["org_review"].notna() & (short_gen["text_len"] > 5)]

print(f"Total generalised: {len(generalised)}")
print(f"Short generalised (< 100 chars): {len(short_gen)}")
print(f"\nShortest examples:")
print(short_gen[["review_id","state_name", "period","east_west" ,"connotation", "text_len", "org_review", "trans_review"]].head(30).to_string())

short_gen.to_csv(BASE + "short_generalised_reviews.csv", index=False)
print("\nSaved to short_generalised_reviews.csv")

In [ ]:
social_reviews= pd.read_parquet('/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/racialised_classified_text_reviews.parquet')
social_reviews = social_reviews[social_reviews['review_id'].isin(all_reviews_df['review_id'].unique())]
social_reviews.info()


In [ ]:
social_reviews.to_parquet('/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/racialised_classified_text_reviews.parquet')

In [ ]:
import pandas as pd
 
# ── Load constituency-level aggregation ──────────────────────────────────
df = pd.read_csv("/home/ethel/Desktop/swimming_pool_project/spp_15052026/openday_2026/constituency_aggregated.csv")   # adjust path as needed
 
# ── Summary stats ─────────────────────────────────────────────────────────
stats = {
    "all": {
        "reviews":  int(df["total_reviews"].sum()),
        "pools":    int(df["pool_count"].sum()),
        "states":   int(df["state_name"].nunique()),
    },
    "social": {
        "reviews":  int(df["social_reviews"].sum()),
        "pools":    int((df["social_reviews"] > 0).sum()),   # pools with ≥1 social review
        "states":   int(df[df["social_reviews"] > 0]["state_name"].nunique()),
    },
    "racialised": {
        "reviews":  int(df["racialised_reviews"].sum()),
        "pools":    int((df["racialised_reviews"] > 0).sum()),
        "states":   int(df[df["racialised_reviews"] > 0]["state_name"].nunique()),
    },
}
 
# ── Print ─────────────────────────────────────────────────────────────────
print(f"{'':20s} {'Reviews':>12} {'Pools':>8} {'States':>8}")
print("-" * 50)
for category, s in stats.items():
    print(f"{category.capitalize():20s} {s['reviews']:>12,} {s['pools']:>8,} {s['states']:>8}")
 

In [ ]:
racialised_reviews_df.unique_identifier.nunique()

In [ ]:
spatial_join_gdf[spatial_join_gdf['racialised'] == 'yes'].state_name.nunique()

In [ ]:
spatial_join_gdf.info()